In [1]:
import requests
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error
from datetime import datetime, timedelta
import pytz

In [2]:
API_key = '66923e50fbec7ab338766b1f94dd3aa7'
BASE_URL = 'https://api.openweathermap.org/data/2.5/' # base url

####  1.fetch current weather data

In [3]:
def get_current_weather(city):
    url = f"{BASE_URL}weather?q={city}&appid={API_key}&units=metric"
    response = requests.get(url)
    data = response.json()
    return {
       'city' : data['name'],
       'current_temp' : round(data['main']['temp']),
       'feels_like' : round(data['main']['feels_like']),
       'temp_min' : round(data['main']['temp_min']),
       'temp_max' : round(data['main']['temp_max']),
       'humidity' : round(data['main']['humidity']),
       'description' : data['weather'][0]['description'],
       'country' : data['sys']['country'],
       'wind_gust_dir' : data['wind']['deg'],
       'pressure' : data['main']['pressure'],
       'Wind_Gust_Speed' : data['wind']['speed'],
    }

####  2.Read historical data

In [4]:
def read_historical_data(filename):
    df = pd.read_csv(filename)
    df = df.dropna()
    df = df.drop_duplicates()
    return df

####  3.prepare data for traning

In [5]:
def prepare_data(data):
    le = LabelEncoder()
    data['WindGustDir'] = le.fit_transform(data['WindGustDir'])
    data['RainTomorrow'] = le.fit_transform(data['RainTomorrow'])

    X = data[['MinTemp','MaxTemp','WindGustDir','WindGustSpeed','Humidity','Pressure','Temp']]
    y = data['RainTomorrow']
    return X, y, le

####  4.Train Rain prediction

In [6]:
def train_rain_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("Mean squared error for rain model:", mean_squared_error(y_test, y_pred))
    return model

#### 5.prepare regression data

In [7]:
def prepare_regression_data_lag(data, feature, lag=3):
    X, y = [], []
    for i in range(lag, len(data)):
        X.append(data[feature].iloc[i-lag:i].values)
        y.append(data[feature].iloc[i])
    return np.array(X), np.array(y)

####  6.train regression data

In [8]:
def train_regression_model_lag(X, y):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X, y)
    return model

#### 7.predict future

In [9]:
def predict_future_lag(model, last_values, steps=5):
    predictions = list(last_values)
    lag = len(last_values)
    for _ in range(steps):
        next_value = model.predict(np.array([predictions[-lag:]]))[0]
        predictions.append(next_value)
    return predictions[lag:]

#### 8.weather analysis Function

In [11]:
def weather_view():
    city = input("Enter any city name: ")
    current_weather = get_current_weather(city)

    # Load historical data
    historical_data = read_historical_data("weather.csv")

    # Train rain prediction model
    X, y, le = prepare_data(historical_data)
    rain_model = train_rain_model(X, y)

    # Wind direction → compass
    wind_deg = current_weather["wind_gust_dir"] % 360
    compass_points = [
        ("N", 0, 11.25), ("NNE", 11.25, 33.75), ("NE", 33.75, 56.25),
        ("ENE", 56.25, 78.75), ("E", 78.75, 101.25), ("ESE", 101.25, 123.75),
        ("SE", 123.75, 146.25), ("SSE", 146.25, 168.75), ("S", 168.75, 191.25),
        ("SSW", 191.25, 213.75), ("SW", 213.75, 236.25), ("WSW", 236.25, 258.75),
        ("W", 258.75, 281.25), ("WNW", 281.25, 303.75), ("NW", 303.75, 326.25),
        ("NNW", 326.25, 348.75), ("N", 348.75, 360)
    ]
    compass_direction = next(
        point for point, start, end in compass_points
        if start <= wind_deg < end
    )
    compass_direction_encoded = le.transform([compass_direction])[0] if compass_direction in le.classes_ else -1

    # Current weather dataframe for rain prediction
    current_data = {
        "MinTemp": current_weather["temp_min"],
        "MaxTemp": current_weather["temp_max"],
        "WindGustDir": compass_direction_encoded,
        "WindGustSpeed": current_weather["Wind_Gust_Speed"],
        "Humidity": current_weather["humidity"],
        "Pressure": current_weather["pressure"],
        "Temp": current_weather["current_temp"],
    }
    current_df = pd.DataFrame([current_data])
    rain_prediction = rain_model.predict(current_df)[0]

    # Regression models with lag
    lag = 3

    # Temperature
    X_temp, y_temp = prepare_regression_data_lag(historical_data, "Temp", lag)
    temp_model = train_regression_model_lag(X_temp, y_temp)
    last_temp_values = historical_data["Temp"].iloc[-lag:].tolist()
    future_temp = predict_future_lag(temp_model, last_temp_values)

    # Humidity
    X_hum, y_hum = prepare_regression_data_lag(historical_data, "Humidity", lag)
    hum_model = train_regression_model_lag(X_hum, y_hum)
    last_hum_values = historical_data["Humidity"].iloc[-lag:].tolist()
    future_humidity = predict_future_lag(hum_model, last_hum_values)

    # Future times
    timezone = pytz.timezone("Asia/Karachi")
    now = datetime.now(timezone)
    next_hour = (now + timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)
    future_times = [(next_hour + timedelta(hours=i)).strftime("%H:00") for i in range(5)]

    # Display output
    print(f"\nCity: {city}, {current_weather['country']}")
    print(f"Current Temperature: {current_weather['current_temp']}°C")
    print(f"Feels Like: {current_weather['feels_like']}")
    print(f"Minimum Temperature: {current_weather['temp_min']}°C")
    print(f"Maximum Temperature: {current_weather['temp_max']}°C")
    print(f"Humidity: {current_weather['humidity']}%")
    print(f"Weather Prediction: {current_weather['description']}")
    print(f"Rain Prediction: {'Yes' if rain_prediction else 'No'}")

    print("\nFuture Temperature Prediction:")
    for time, temp in zip(future_times, future_temp):
        print(f"{time}: {round(temp, 1)}°C")

    print("\nFuture Humidity Prediction:")
    for time, humidity in zip(future_times, future_humidity):
        print(f"{time}: {round(humidity, 1)}%")

# Run
weather_view()

Enter any city name:  england


Mean squared error for rain model: 0.1506849315068493

City: england, US
Current Temperature: 13°C
Feels Like: 12
Minimum Temperature: 12°C
Maximum Temperature: 13°C
Humidity: 70%
Weather Prediction: broken clouds
Rain Prediction: No

Future Temperature Prediction:
22:00: 21.8°C
23:00: 21.1°C
00:00: 21.3°C
01:00: 20.7°C
02:00: 20.9°C

Future Humidity Prediction:
22:00: 41.1%
23:00: 46.9%
00:00: 40.8%
01:00: 43.0%
02:00: 41.2%
